# Aula 11 — Treinar e Usar um LLM

Três exercícios, um depois de cada microaula. Rode as células na ordem.
Troque cada `____` pelo que falta e rode; a célula seguinte confere: ✅
deu certo, ❌ traz uma pista. As células 🧰 são código pronto: só rode.

**Hoje é o dia do treino.** No fim do Exercício 2 você liga um treino de
35 minutos, que roda durante o intervalo e a terceira microaula. Não
feche a aba enquanto ele roda.

**Travou?** [Colas dos exercícios](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios).

## Preparação

🧰 O tokenizador da Aula 9, o texto de Machado em tokens, o
`sortear_lote`, o modelo que você montou na Aula 10 (`montar_modelo`) e
o modelo treinado do professor (`professor`). Leva uns 30 segundos.

In [ ]:
import collections
import json
import math
import os
import re
import time
import urllib.request
import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import ops

keras.utils.set_random_seed(0)
warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"


def baixar(nome, binario=False):
    local = os.path.join("..", "..", "data", nome)   # só existe no PC do professor
    if os.path.exists(local):
        if binario:
            return open(local, "rb").read()
        return open(local, encoding="utf-8").read()
    dados = urllib.request.urlopen(URL_DADOS + nome).read()
    if binario:
        return dados
    return dados.decode("utf-8")


# --- Tokenizador (Aula 9) ---
oficial = json.loads(baixar("machado_bpe.json"))
PADRAO = re.compile(oficial["padrao"])
fusoes = {}
for k, (a, b) in enumerate(oficial["fusoes"]):
    fusoes[(a, b)] = 256 + k
tabela = {}
for i in range(256):
    tabela[i] = bytes([i])
for (a, b), novo in fusoes.items():
    tabela[novo] = tabela[a] + tabela[b]


def contar_pares(ids):
    pares = collections.Counter()
    for par in zip(ids, ids[1:]):
        pares[par] += 1
    return pares


def fundir(ids, par, novo):
    saida = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == par[0] and ids[i + 1] == par[1]:
            saida.append(novo)
            i = i + 2
        else:
            saida.append(ids[i])
            i = i + 1
    return saida


def ordem(par):
    return fusoes.get(par, 10**9)


def codificar_palavra(palavra):
    ids = list(palavra.encode("utf-8"))
    while len(ids) >= 2:
        par = min(contar_pares(ids), key=ordem)
        if par not in fusoes:
            break
        ids = fundir(ids, par, fusoes[par])
    return ids


memoria = {}


def codificar(texto):
    saida = []
    for palavra in PADRAO.findall(texto):
        if palavra not in memoria:
            memoria[palavra] = codificar_palavra(palavra)
        saida.extend(memoria[palavra])
    return saida


def decodificar(ids):
    juntos = b""
    for i in ids:
        juntos = juntos + tabela[int(i)]
    return juntos.decode("utf-8", errors="replace")


def mostrar(i):
    return decodificar([i]).replace(" ", "·")       # o espaço aparece como ·


# --- O texto em tokens e o lote (Aula 10) ---
TAMANHO_VOCABULARIO = 1024
BLOCO = 128
LOTE = 32

tokens = np.array(codificar(baixar("machado.txt")))
corte = int(0.9 * len(tokens))
dados_treino = tokens[:corte]
dados_validacao = tokens[corte:]


def sortear_lote(dados):
    inicios = np.random.randint(0, len(dados) - BLOCO - 1, LOTE)
    entradas = []
    alvos = []
    for inicio in inicios:
        entradas.append(dados[inicio : inicio + BLOCO])
        alvos.append(dados[inicio + 1 : inicio + BLOCO + 1])
    return np.stack(entradas), np.stack(alvos)


def lotes(dados):
    while True:
        yield sortear_lote(dados)


# --- O modelo (Aula 10) ---
DIMENSAO = 128
CABECAS = 4
CAMADAS = 4
OCULTA = 256


class TokenEPosicao(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.token = keras.layers.Embedding(TAMANHO_VOCABULARIO, DIMENSAO, name="token")
        self.posicao = keras.layers.Embedding(BLOCO, DIMENSAO, name="posicao")

    def call(self, ids):
        n = ops.shape(ids)[1]
        return self.token(ids) + self.posicao(ops.arange(n))


def bloco(x, i):
    h = keras.layers.LayerNormalization(name=f"norma_a{i}")(x)
    h = keras.layers.MultiHeadAttention(num_heads=CABECAS, key_dim=DIMENSAO // CABECAS,
                                        name=f"atencao{i}")(h, h, use_causal_mask=True)
    x = x + h
    h = keras.layers.LayerNormalization(name=f"norma_b{i}")(x)
    h = keras.layers.Dense(OCULTA, activation="gelu", name=f"densa{i}")(h)
    h = keras.layers.Dense(DIMENSAO, name=f"volta{i}")(h)
    return x + h


def montar_modelo():
    entrada = keras.Input(shape=(None,), dtype="int32", name="tokens")
    x = TokenEPosicao(name="embedding")(entrada)
    for i in range(CAMADAS):
        x = bloco(x, i)
    x = keras.layers.LayerNormalization(name="norma_final")(x)
    saida = keras.layers.Dense(TAMANHO_VOCABULARIO, name="saida")(x)
    return keras.Model(entrada, saida)


open("mini_llm.weights.h5", "wb").write(baixar("mini_llm.weights.h5", binario=True))
professor = montar_modelo()
professor.load_weights("mini_llm.weights.h5")


def amostra(modelo, texto="A casa de", quantos=40):
    # gera texto sorteando com as chances do modelo (no Exercício 2 você escreve a sua)
    indices = codificar(texto)
    for _ in range(quantos):
        logits = modelo(np.array([indices[-BLOCO:]]))
        indices.append(int(keras.random.categorical(logits[:, -1] / 0.8, 1)[0, 0]))
    return decodificar(indices)


print(f"{len(tokens):,} tokens de texto e o modelo do professor prontos.".replace(",", "."))

---
## Exercício 1: o treino

📝 Antes: [quiz da Microaula 1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-1-o-treino).
🆘 Travou? [Cola do Exercício 11.1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-1)

🎯 A função que treina o seu LLM pelo relógio e guarda sozinha o melhor
modelo, testada por um minuto.

### 1.a O relógio

🧰 `taxa_no_momento(fracao)` dá o tamanho do passo conforme a fração do
tempo que já passou: sobe no começo (aquecimento) e desce até o fim
(resfriamento). O `Relogio` é um *callback*: o `fit` chama ele a cada
passo, e ele ajusta a taxa, para o treino quando o tempo acaba e anota a
curva.

In [ ]:
TAXA_MAXIMA = 1e-3


def taxa_no_momento(fracao):
    if fracao < 0.03:
        return TAXA_MAXIMA * fracao / 0.03                                     # aquecimento
    resto = (fracao - 0.03) / 0.97
    return TAXA_MAXIMA * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * resto)))   # resfriamento


class Relogio(keras.callbacks.Callback):
    def __init__(self, minutos):
        super().__init__()
        self.minutos = minutos
        self.historico = {"minuto": [], "treino": [], "validacao": []}

    def on_train_begin(self, logs=None):
        self.inicio = time.time()

    def on_train_batch_begin(self, passo, logs=None):
        fracao = (time.time() - self.inicio) / (self.minutos * 60)
        if fracao >= 1:
            self.model.stop_training = True
        self.model.optimizer.learning_rate.assign(taxa_no_momento(min(fracao, 1)))

    def on_epoch_end(self, conferencia, logs=None):
        minuto = (time.time() - self.inicio) / 60
        self.historico["minuto"].append(minuto)
        self.historico["treino"].append(logs["loss"])
        self.historico["validacao"].append(logs["val_loss"])
        print(f"conferência {conferencia + 1:>3}   {minuto:4.1f} min   "
              f"treino {logs['loss']:.2f}   validação {logs['val_loss']:.2f}")


fracoes = np.linspace(0, 1, 101)
plt.figure(figsize=(8, 3))
plt.plot(fracoes * 35, [taxa_no_momento(f) for f in fracoes])
plt.xlabel("minutos de treino")
plt.ylabel("tamanho do passo")
plt.show()

### 1.b A função de treino

O `compile` escolhe o otimizador (AdamW, com corte do gradiente) e a
perda. O `fit` repete o ciclo: sorteia um lote, prevê, mede a perda, corrige os
pesos. A cada 200 passos ele confere a perda na validação, e o
`ModelCheckpoint` salva o modelo em `meu_llm.weights.h5` sempre que ela
bate o recorde. Duas lacunas: de onde vêm os lotes da validação, e o
nome que o Keras dá à perda na validação. No fim, um teste de **1 minuto**.

In [ ]:
def treinar(modelo, minutos):
    modelo.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=TAXA_MAXIMA, weight_decay=0.1,
                                         beta_2=0.95, global_clipnorm=1.0),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    relogio = Relogio(minutos)
    salvar_o_melhor = keras.callbacks.ModelCheckpoint(
        "meu_llm.weights.h5", monitor="____", save_best_only=True, save_weights_only=True)
    try:
        modelo.fit(lotes(dados_treino), steps_per_epoch=200, epochs=100_000,
                   validation_data=lotes(____), validation_steps=10,
                   callbacks=[relogio, salvar_o_melhor], verbose=0)
    except KeyboardInterrupt:
        print("Treino interrompido por você. O melhor modelo já está salvo.")
    print(f"Fim: melhor validação {min(relogio.historico['validacao']):.2f}")
    return relogio.historico


teste = montar_modelo()
print("antes: ", amostra(teste))
historico_teste = treinar(teste, minutos=1)
print("depois:", amostra(teste))

In [ ]:
if historico_teste["validacao"] and historico_teste["validacao"][-1] < 6.0 and os.path.exists("meu_llm.weights.h5"):
    print("✅ Em um minuto a perda já saiu de 6,9, e o melhor modelo foi salvo. "
          "O treino de verdade é este mesmo, por 35 minutos, no fim do Exercício 2.")
else:
    print('❌ A validação vem de lotes(dados_validacao), e o recorde é medido em monitor="val_loss".')

✏️ Um minuto de treino já mudou o texto. O que apareceu primeiro: as
palavras, os espaços ou a pontuação? →

---
## Exercício 2: do número à palavra

📝 Antes: [quiz da Microaula 2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-2-do-numero-a-palavra).
🆘 Travou? [Cola do Exercício 11.2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-2)

🎯 A sua função `gerar`, com os dois botões de toda ferramenta de IA:
temperatura e top-k. Ela usa o modelo do professor enquanto o seu ainda
não existe. No fim, você liga o seu treino.

### 2.a A última posição

O modelo devolve 1.024 pontuações para **cada** posição do texto. Para
continuar o texto, só interessa a última.

In [ ]:
texto = "Não sei se a senhora"
logits = professor(np.array([codificar(texto)]))
print("o modelo devolveu:", logits.shape, "(1 frase, posições, 1.024 pontuações)")

ultimos = logits[0, ____]
chances = np.asarray(ops.softmax(ultimos))

for i in np.argsort(chances)[::-1][:10]:
    print(f"{mostrar(i):>10}  {chances[i]:6.1%}")
print(f"as 10 melhores juntas: {np.sort(chances)[-10:].sum():.0%}")

In [ ]:
if tuple(ultimos.shape) == (1024,) and np.allclose(np.asarray(ultimos), np.asarray(logits[0, len(codificar(texto)) - 1])):
    print("✅ Uma distribuição sobre 1.024 tokens. Nenhuma resposta certa: só chances.")
else:
    print("❌ A última posição é a de índice -1: logits[0, -1].")

### 2.b A temperatura

`keras.random.categorical(pontuacoes, 1)` sorteia um token com as chances
que as pontuações dão. Antes de sortear, divida as pontuações pela
temperatura `T`:

$$P_j = \frac{e^{z_j / T}}{\sum_{k} e^{z_k / T}}$$

Temperatura baixa concentra a chance no favorito; alta espalha.

In [ ]:
def gerar(modelo, texto, quantos=60, temperatura=0.8):
    indices = codificar(texto)
    for _ in range(quantos):
        logits = modelo(np.array([indices[-BLOCO:]]))
        ultimos = logits[:, -1]
        sorteado = keras.random.categorical(ultimos / ____, 1)
        indices.append(int(sorteado[0, 0]))
    return decodificar(indices)


for temperatura in [0.3, 0.8, 1.5]:
    print(f"T = {temperatura}:", gerar(professor, "A casa de", temperatura=temperatura))
    print("---")

In [ ]:
keras.utils.set_random_seed(1)
fria = [gerar(professor, "Capitú", 8, temperatura=0.0001) for _ in range(3)]
quentes = [gerar(professor, "Capitú", 8, temperatura=1.5) for _ in range(3)]
if len(set(fria)) == 1 and len(set(quentes)) > 1:
    print("✅ Perto de zero, o sorteio sempre tira o favorito: três vezes o mesmo texto. "
          "Com 1,5, três textos diferentes.")
else:
    print("❌ Divida as pontuações pela temperatura: ultimos / temperatura.")

### 2.c O top-k

A temperatura reescala tudo, inclusive os tokens ruins. O top-k corta a
cauda: só os `k` mais prováveis entram no sorteio. `ops.top_k` devolve as
`k` maiores pontuações e de quais tokens elas são.

In [ ]:
def gerar(modelo, texto, quantos=60, temperatura=0.8, k=40):
    indices = codificar(texto)
    for _ in range(quantos):
        logits = modelo(np.array([indices[-BLOCO:]]))
        ultimos = logits[0, -1]
        melhores, candidatos = ops.top_k(ultimos, ____)
        sorteado = keras.random.categorical(melhores[None] / temperatura, 1)
        indices.append(int(candidatos[int(sorteado[0, 0])]))
    return decodificar(indices)


for k in [1, 40, 1024]:
    print(f"k = {k}:", gerar(professor, "A casa de", k=k))
    print("---")

In [ ]:
um = gerar(professor, "Capitú", 10, k=1)
outro = gerar(professor, "Capitú", 10, k=1)
if um == outro:
    print("✅ Com k = 1 só o favorito entra no sorteio: é a escolha gulosa, sempre o mesmo texto. "
          "Com k = 1024 não se corta nada.")
else:
    print("❌ O número de candidatos é o k: ops.top_k(ultimos, k).")

✏️ O texto com k = 1 entrou em loop, repetindo um pedaço? Por quê? →

### 2.d Ligue o seu treino (antes do intervalo!)

São 35 minutos. A cada 200 passos aparece uma linha com a perda no
treino e na validação, e o melhor modelo fica salvo em
`meu_llm.weights.h5`. Para parar antes, aperte o ■ da célula: o melhor
modelo continua salvo.

In [ ]:
keras.utils.set_random_seed(42)
modelo = montar_modelo()
historico = treinar(modelo, minutos=35)

---
## Exercício 3: o seu modelo

📝 Antes: [quiz da Microaula 3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-11-treinar-e-usar#microaula-3-ajuste-fino-e-limites).
🆘 Travou? [Cola do Exercício 11.3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-11-3)

🎯 O seu LLM treinado escrevendo, e depois ajustado para escrever sobre
outra coisa.

### 3.a A curva e o melhor modelo

O arquivo guarda o modelo da **melhor validação**, não o último.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(historico["minuto"], historico["treino"], label="treino")
plt.plot(historico["minuto"], historico["validacao"], label="validação")
plt.axhline(math.log(1024), color="gray", linestyle=":", label="chute ao acaso")
plt.xlabel("minutos de treino")
plt.ylabel("perda")
plt.legend()
plt.show()

modelo.load_weights("____")
perda_final = modelo.evaluate(lotes(dados_validacao), steps=10, verbose=0)
print(f"perda na validação: {perda_final:.2f}   perplexidade: {math.exp(perda_final):.0f} (antes: 1024)")

In [ ]:
if perda_final < 4.2:
    print(f"✅ O seu modelo está na dúvida entre uns {math.exp(perda_final):.0f} tokens, e não mais entre 1.024.")
else:
    print("❌ Confira o nome do arquivo: o treino salvou em meu_llm.weights.h5. "
          "Se ele parou muito cedo, a perda ainda pode estar alta: fale com o professor.")

### 3.b O seu modelo escreve

🧰 A sua `gerar`, agora com o **seu** modelo.

In [ ]:
for comeco in ["A casa de", "Não sei se a senhora", "Capitú"]:
    print(gerar(modelo, comeco))
    print("---")

### 3.c Ajuste fino: o mesmo treino, outro texto

🧰 O texto deste curso (as páginas do livro), em tokens, separado em
treino e validação.

In [ ]:
tokens_curso = np.array(codificar(baixar("curso.txt")))
corte_curso = int(0.9 * len(tokens_curso))
curso_treino = tokens_curso[:corte_curso]
print(f"{len(tokens_curso):,} tokens de texto do curso.".replace(",", "."))

Uma cópia do seu modelo continua o treino por 200 passos, com um passo
menor (para mexer pouco no que ele já sabe). A única coisa que muda é de
onde vêm os lotes.

In [ ]:
ajustado = montar_modelo()
ajustado.set_weights(modelo.get_weights())                 # uma cópia do seu modelo
ajustado.compile(optimizer=keras.optimizers.AdamW(learning_rate=3e-4, global_clipnorm=1.0),
                 loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
ajustado.fit(lotes(____), steps_per_epoch=50, epochs=4)

In [ ]:
def estranheza(modelo, frase):
    # a perda de uma frase só: baixa quer dizer que o modelo acha a frase provável
    ids = np.array([codificar(frase)])
    return float(modelo.evaluate(ids[:, :-1], ids[:, 1:], verbose=0))


modelo.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
frases = ["Não sei se a senhora sabe que elle não veio hontem.",
          "O modelo de linguagem prevê o próximo token do texto."]
print(f"{'frase':<56} {'original':>9} {'ajustado':>9}")
for frase in frases:
    print(f"{frase:<56} {estranheza(modelo, frase):>9.2f} {estranheza(ajustado, frase):>9.2f}")

if estranheza(ajustado, frases[1]) < estranheza(modelo, frases[1]) - 1:
    print("\n✅ A frase de hoje ficou muito menos estranha para o modelo ajustado. Nenhuma "
          "linha da arquitetura mudou: só o texto do treino.")
else:
    print("\n❌ Os lotes do ajuste vêm do texto do curso: lotes(curso_treino).")

🧰 O mesmo começo, nos dois modelos:

In [ ]:
for nome, m in [("original", modelo), ("ajustado", ajustado)]:
    print(f"{nome}: {gerar(m, 'O modelo')}")
    print("---")

✏️ Do que o modelo ajustado passou a falar? E ele sabe se o que diz é
verdade? →

### 3.d Leve o seu modelo para casa

O Colab apaga tudo quando a aba fecha. Baixe o `meu_llm.weights.h5`
(uns 3 MB). Ele carrega em qualquer `montar_modelo()` com
`modelo.load_weights("meu_llm.weights.h5")`.

In [ ]:
try:
    from google.colab import files
    files.download("meu_llm.weights.h5")
    print("Download iniciado: confira a pasta de downloads do seu computador.")
except ImportError:
    print("Fora do Colab: o arquivo meu_llm.weights.h5 está na mesma pasta deste notebook.")

---
## O que você construiu hoje

| Peça | Função |
|---|---|
| `Relogio` e `treinar` | o treino pelo relógio, que guarda o melhor modelo sozinho |
| `gerar` | do número à palavra, com temperatura e top-k |
| `ajustado` | o mesmo modelo, ajustado para outro texto em 200 passos |
| `meu_llm.weights.h5` | **o seu modelo de linguagem, treinado** |